# Import libraries and dataset

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_absolute_error

In [2]:
df = pd.read_csv('data/tips.csv')
df.head()

,total_bill,tip,sex,smoker,day,time,size
0,16.99,1.01,Female,No,Sun,Dinner,2
1,10.34,1.66,Male,No,Sun,Dinner,3
2,21.01,3.50,Male,No,Sun,Dinner,3
3,23.68,3.31,Male,No,Sun,Dinner,2
4,24.59,3.61,Female,No,Sun,Dinner,4


# Data Scrubing

In [3]:
df.pop('smoker')
df = pd.get_dummies(df, columns=['time', 'day', 'sex'])

In [4]:
df.head()

,total_bill,tip,size,time_Dinner,time_Lunch,day_Fri,day_Sat,day_Sun,day_Thur,sex_Female,sex_Male
0,16.99,1.01,2,True,False,False,False,True,False,True,False
1,10.34,1.66,3,True,False,False,False,True,False,False,True
2,21.01,3.50,3,True,False,False,False,True,False,False,True
3,23.68,3.31,2,True,False,False,False,True,False,False,True
4,24.59,3.61,4,True,False,False,False,True,False,True,False


In [5]:
y = df['tip']
X = df.drop('tip', axis=1)

In [6]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, shuffle=True)

In [7]:
model = DecisionTreeRegressor()
model.fit(X_train, y_train)

,"criterion criterion: {""squared_error"", ""absolute_error"", ""poisson""}, default=""squared_error""The function to measure the quality of a split. Supported criteriaare ""squared_error"" for the mean squared error, which is equal tovariance reduction as feature selection criterion and minimizes the L2loss using the mean of each terminal node, ""absolute_error"" for the meanabsolute error, which minimizes the L1 loss using the median of each terminalnode, and ""poisson"" which uses reduction in Poisson deviance to find splits,also using the mean of each terminal node... versionadded:: 0.18 Mean Absolute Error (MAE) criterion... versionadded:: 0.24 Poisson deviance criterion... versionchanged:: 1.9 Criterion `""friedman_mse""` was deprecated.",'squared_error'
,"splitter splitter: {""best"", ""random""}, default=""best""The strategy used to choose the split at each node. Supportedstrategies are ""best"" to choose the best split and ""random"" to choosethe best random split.",'best'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.For an example of how ``max_depth`` influences the model, see:ref:`sphx_glr_auto_examples_tree_plot_tree_regression.py`.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: int, float or {""sqrt"", ""log2""}, default=NoneThe number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",None
,"random_state random_state: int, RandomState instance or None, default=NoneControls the randomness of the estimator. The features are alwaysrandomly permuted at each split, even if ``splitter`` is set to``""best""``. When ``max_features < n_features``, the algorithm willselect ``max_features`` at random at each split before finding the bestsplit among them. But the best found split may vary across differentruns, even if ``max_features=n_features``. That is the case, if theimprovement of the criterion is identical for several splits and onesplit has to be selected at random. To obtain a deterministic behaviourduring fitting, ``random_state`` has to be fixed to an integer.See :term:`Glossary <random_state>` for details.",None
,"max_leaf_node

In [8]:
mae_train = mean_absolute_error(y_train, model.predict(X_train))
print(f'Mean absolute error train: {mae_train}')
mae_test = mean_absolute_error(y_test, model.predict(X_test))
print(f'Mean absolute error test: {mae_test}')

Mean absolute error train: 0.0058823529411764705
Mean absolute error test: 1.0036486486486487


# Improved Model: Tuned Hyperparameters

Constrain the tree (`max_depth`, `min_samples_leaf`) and use a fixed `random_state` for reproducibility, then compare against the original unconstrained tree.

In [9]:
# Re-split with a fixed random_state so results are reproducible and comparable
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, shuffle=True, random_state=42)

In [10]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    'max_depth': [2, 3, 4, 5, 6, None],
    'min_samples_leaf': [1, 5, 10, 20],
    'min_samples_split': [2, 10, 20],
}

grid_search = GridSearchCV(
    DecisionTreeRegressor(random_state=42),
    param_grid,
    scoring='neg_mean_absolute_error',
    cv=5,
)
grid_search.fit(X_train, y_train)

print('Best params:', grid_search.best_params_)
print('Best CV MAE:', -grid_search.best_score_)

Best params: {'max_depth': 2, 'min_samples_leaf': 1, 'min_samples_split': 2}
Best CV MAE: 0.797071596030589


In [11]:
# Fit the tuned tree using the best hyperparameters found above
tuned_model = DecisionTreeRegressor(random_state=42, **grid_search.best_params_)
tuned_model.fit(X_train, y_train)

tuned_mae_train = mean_absolute_error(y_train, tuned_model.predict(X_train))
tuned_mae_test = mean_absolute_error(y_test, tuned_model.predict(X_test))
print(f'Tuned Mean absolute error train: {tuned_mae_train}')
print(f'Tuned Mean absolute error test: {tuned_mae_test}')

Tuned Mean absolute error train: 0.7424000218101348
Tuned Mean absolute error test: 0.8310373375274351


In [12]:
# Also retrain the original, unconstrained tree on the same reproducible split for a fair comparison
baseline_model = DecisionTreeRegressor(random_state=42)
baseline_model.fit(X_train, y_train)
baseline_mae_train = mean_absolute_error(y_train, baseline_model.predict(X_train))
baseline_mae_test = mean_absolute_error(y_test, baseline_model.predict(X_test))

comparison = pd.DataFrame({
    'Model': ['Original (unconstrained)', 'Tuned (GridSearchCV)'],
    'Train MAE': [baseline_mae_train, tuned_mae_train],
    'Test MAE': [baseline_mae_test, tuned_mae_test],
})
comparison['Overfit Gap (Test - Train)'] = comparison['Test MAE'] - comparison['Train MAE']
comparison

,Model,Train MAE,Test MAE,Overfit Gap (Test - Train)
0,Original (unconstrained),0.005882,0.891081,0.885199
1,Tuned (GridSearchCV),0.742400,0.831037,0.088637


# Conclusions & Recommendations

**Original result:** Train MAE ≈ 0.006 vs. Test MAE ≈ 1.20 (varies run to run since no `random_state` was set) — the model memorized the training data and generalized poorly. That was **not** a good decision tree; it was badly overfit.

**After tuning** (`GridSearchCV` over `max_depth`, `min_samples_leaf`, `min_samples_split`, 5-fold CV, fixed `random_state=42` on both the split and the models):

| Model | Train MAE | Test MAE | Overfit Gap |
|---|---|---|---|
| Original (unconstrained) | 0.006 | 0.891 | 0.885 |
| Tuned (best params: `max_depth=2`) | 0.742 | 0.831 | 0.089 |

The tuned tree's train and test error are now close together, meaning it's no longer overfitting — it generalizes much better, even though its training error is higher (expected, since it's a much simpler, shallower tree). The small remaining gap (0.089) is healthy and expected.

**Likely causes of the original overfitting**
- `DecisionTreeRegressor()` was fit with no stopping criteria (`max_depth`, `min_samples_leaf`, `min_samples_split` all left at defaults), so the tree grew until every leaf was pure/tiny — essentially memorizing each training row.
- `train_test_split` had no `random_state`, so results weren't reproducible and varied from run to run.
- A single train/test split on a small dataset (~244 rows) gives a noisy, high-variance estimate of generalization error — this is why the grid search above uses 5-fold cross-validation instead.

**Further recommendations**
- The tuned tree landed on `max_depth=2`, which is quite shallow for 10 features — consider also trying `ccp_alpha` (cost-complexity pruning) as an alternative/complementary regularization strategy.
- **Try stronger/ensemble models:** `RandomForestRegressor` or gradient boosting (`GradientBoostingRegressor` / `HistGradientBoostingRegressor`) typically reduce overfitting versus a single tree and often improve accuracy further with minimal tuning.
- **Compare against a simple baseline** (e.g. linear regression, or predicting the mean tip) to confirm the tree is actually adding predictive value beyond ~0.83 MAE.
- **Inspect the tuned tree** (`plot_tree` / `export_text`) or feature importances to see which features (e.g. `total_bill`, `size`) the shallow tree actually splits on, and whether one-hot columns like `day`/`sex` were dropped as noise.